# Streaming — 流式输出

对标文档：[LangChain > Core Components > Streaming](https://langchain-doc.cn/v1/python/langchain/overview.html)

In [1]:
from langchain_learning.llm import get_llm
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()

**术语：**
- **Stream（流式）** = LLM 边生成边返回结果，不用等全部生成完
- **chunk** = 每次返回的一小段内容，多个 chunk 拼起来就是完整回复
- **异步（async/await）** = 不阻塞主线程的写法，适合高并发场景

### 1. stream() 基本用法

In [2]:
print("开始流式输出：")
for chunk in llm.stream("写一首短诗，关于编程"):
    print(chunk.content, end="", flush=True)
print()

开始流式输出：
## 《代码》

我写下第一行代码
屏幕亮起
光标在黑暗中闪烁
像一只萤火虫
寻找它的同类

我写下第二行代码
循环开始
数据在内存中游走
像一群候鸟
寻找它们的季节

我写下第三行代码
函数返回
错误在日志里堆积
像秋天的落叶
等待一场清理

我写下第四行代码
程序运行
世界在二进制中重组
像初生的宇宙
等待它的第一个黎明


**术语：**
- **stream()** = 返回一个迭代器，每次 yield 一个 chunk
- **flush=True** = 强制刷新输出缓冲区，让文字实时显示
- invoke 和 stream 的区别：invoke 等全部生成完一次性返回，stream 边生成边返回

### 2. 在 LCEL 链中使用 stream

In [3]:
chain = ChatPromptTemplate.from_template("用一句话介绍{topic}") | llm | StrOutputParser()

print("流式输出：")
for chunk in chain.stream({"topic": "LangChain"}):
    print(chunk, end="", flush=True)
print()

流式输出：
LangChain是一个用于构建基于大语言模型（LLM）的应用程序的开源框架，通过提供模块化组件（如链、代理、记忆和工具集成）来简化与外部数据源和API的交互。


**术语：**
- 支持 stream 的链叫 **可流式链（streamable）**
- StrOutputParser 支持流式：逐步把 LLM 的 chunk 转为文本
- PydanticOutputParser **不支持**流式（需要完整的 JSON 才能解析）

### 3. 异步流式（async / await）

In [4]:
async def async_stream():
    chain = ChatPromptTemplate.from_template("解释什么是{topic}，50字以内") | llm | StrOutputParser()
    print("异步流式：")
    async for chunk in chain.astream({"topic": "机器学习"}):
        print(chunk, end="", flush=True)
    print()

await async_stream()

异步流式：
机器学习是让计算机从数据中自动学习规律，无需明确编程，从而进行预测或决策。


**术语：**
- **astream** = stream 的异步版本，用 `async for` 遍历
- **async / await** = Python 异步编程，适合 I/O 密集场景
- 在 Jupyter 中也可以直接用 `await chain.astream(...)`

```mermaid
graph LR
    A[输入] --> B[PromptTemplate]
    B --> C[LLM]
    C -.->|chunk 1| D[StrOutputParser]
    C -.->|chunk 2| D
    C -.->|chunk 3| D
    D -.->|实时输出| E[终端]
    style C fill:#FFF3E0,stroke:#E65100
    style D fill:#E8F5E9,stroke:#2E7D32
```

### 4. 事件流（Event Stream）

In [5]:
async def show_events():
    print("事件流：")
    async for event in chain.astream_events({"topic": "人工智能"}, version="v2"):
        if event["event"] == "on_parser_end":
            print(f"[完成] 生成最终结果: {event['data']['output'][:50]}...")
        elif event["event"] == "on_llm_end":
            print("[LLM 完成生成]")
        elif event["event"] == "on_llm_start":
            print("[LLM 开始生成]")

await show_events()

事件流：
[完成] 生成最终结果: 人工智能是模拟人类智能的技术，使机器能够学习、推理、感知和决策。...


**术语：**
- **astream_events（异步事件流）** = 异步返回链执行过程中的所有事件
- **version="v2"** = 指定事件格式版本，新版 LangChain 必须加此参数
- 适合需要**监控执行过程**的场景，比如显示进度条
- 事件名：`on_llm_start`、`on_llm_end`、`on_parser_end` 等

### 5. 自定义回调——实时显示 token 数量

In [6]:
class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.count = 0

    def on_llm_new_token(self, token: str, **kwargs):
        self.count += 1
        print(f"[收到第 {self.count} 个 token]")

counter = TokenCounter()

print("带 token 计数的流式输出：")
for chunk in llm.stream("从 1 数到 3", config={"callbacks": [counter]}):
    print(chunk.content, end="", flush=True)
print(f"\n共收到 {counter.count} 个 token")

带 token 计数的流式输出：
[收到第 1 个 token]
[收到第 2 个 token]
好的[收到第 3 个 token]
，[收到第 4 个 token]
我们[收到第 5 个 token]
开始[收到第 6 个 token]
从[收到第 7 个 token]
1[收到第 8 个 token]
数[收到第 9 个 token]
到[收到第 10 个 token]
3[收到第 11 个 token]
：

[收到第 12 个 token]
**[收到第 13 个 token]
1[收到第 14 个 token]
**[收到第 15 个 token]
  
[收到第 16 个 token]
**[收到第 17 个 token]
2[收到第 18 个 token]
**[收到第 19 个 token]
  
[收到第 20 个 token]
**[收到第 21 个 token]
3[收到第 22 个 token]
**

[收到第 23 个 token]
数[收到第 24 个 token]
完了[收到第 25 个 token]
。[收到第 26 个 token]
如果你[收到第 27 个 token]
有其他[收到第 28 个 token]
问题[收到第 29 个 token]
或[收到第 30 个 token]
需要[收到第 31 个 token]
进一步[收到第 32 个 token]
帮助[收到第 33 个 token]
，[收到第 34 个 token]
请[收到第 35 个 token]
告诉我[收到第 36 个 token]
！[收到第 37 个 token]
[收到第 38 个 token]

共收到 38 个 token


**术语：**
- **CallbackHandler（回调处理器）** = 可以监听执行过程中的各种事件
- **on_llm_new_token** = 每次收到一个新的 token 就会触发
- **config={"callbacks": [...]}** = 把回调挂到这次调用上
- 回调常用于：日志记录、token 计数、实时监控